In [1]:
# 将我们的知识库进行嵌入并保存到向量数据库中
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
model = SentenceTransformer(r"D:\PycharmProjects\pyjupyter\深度学习\深度学习代码\RAG检索增强生成\model\BAAI\bge-large-zh-v1___5")
# 创建向量数据库
# client = chromadb.Client() 存入内存中速度快但是用完就销毁了
client = chromadb.PersistentClient("./chromadb") # 这种方式是永久型数据库

#创建向量数据库集合
collection = client.get_or_create_collection(
    name = "zhuzhu",
    metadata={
        "hhsw:space" : "cosine"
    }
)

In [20]:
# 加载文件
def text_2db():
    path_list = list(Path("knowledge").glob("*.txt"))
    text_list = [] # 文本内容
    
    for path in path_list:
        text = path.read_text(encoding = "utf-8")
        text_list.append(text)
    # 进行文本嵌入
    embedding = model.encode(text_list)
    
    # 存入数据库
    collection.add(
        embeddings = embedding.tolist(), # 向量，这一项必须要写
        documents = text_list,  # 文本，这一项必须要写
        ids = [f"doc_{i}" for i,_ in enumerate(text_list)]# 文件ID，这一项必须要写
    )

In [21]:
# 这个只用执行一次就可以了
text_2db()

数据库中的数据量为4


In [ ]:
query = "健康"
query_embedding = model.encode(query)
data = collection.query(query_embedding.tolist(),n_results=3)

# 使用chromadb时，这个向量距离的计算是 distance = 1-相似度，这里要注意一下。